# 17 — Profiling, gradient accumulation, AMP, and compilation

**Level:** Advanced / expert practice · **Study time:** 90–120 minutes (including experiments)

**Prerequisites:** Notebooks 06, 15–16.

## What you will be able to do
- Prove microbatch accumulation matches a full-batch update.
- Profile CPU operations and benchmark with warmup.
- Try guarded CUDA mixed precision and understand compilation overhead.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 16](16_custom_autograd_and_debugging.ipynb) · [Next: 18](18_distributed_training.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Accumulate gradients correctly

To simulate a batch of N examples, accumulate each microbatch's mean loss multiplied by its size/N. This handles an uneven final microbatch. Do not clear gradients between microbatches. Equivalence requires operations whose results do not depend on batch composition; BatchNorm and stochastic layers can break this comparison.


In [ ]:
import copy
X, y = torch.randn(23, 16), torch.randn(23, 1)
full = nn.Sequential(nn.Linear(16, 32), nn.Tanh(), nn.Linear(32, 1))
micro = copy.deepcopy(full)
opt_full = torch.optim.SGD(full.parameters(), lr=0.01)
opt_micro = torch.optim.SGD(micro.parameters(), lr=0.01)
opt_full.zero_grad(set_to_none=True)
nn.functional.mse_loss(full(X), y).backward(); opt_full.step()
opt_micro.zero_grad(set_to_none=True)
for xb, yb in zip(X.split(6), y.split(6)):
    loss = nn.functional.mse_loss(micro(xb), yb) * len(xb) / len(X)
    loss.backward()
opt_micro.step()
for a, b in zip(full.parameters(), micro.parameters()):
    assert torch.allclose(a, b, atol=1e-6)
print("Full-batch and accumulated updates agree.")


## Step 3 — Measure before optimizing

A profiler identifies time spent inside operations; a benchmark estimates repeated runtime. Use a warmup and repeated measurements. On CUDA, synchronize before reading wall time because launches are asynchronous. These tiny CPU measurements describe this machine, not a universal speed claim.


In [ ]:
from torch.profiler import profile, ProfilerActivity
from torch.utils.benchmark import Timer
full.eval()
with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    with torch.inference_mode():
        for _ in range(10):
            full(X)
print(prof.key_averages().table(sort_by='self_cpu_time_total', row_limit=6))
@torch.inference_mode()
def infer():
    return full(X)
for _ in range(10):
    infer()
measurement = Timer(stmt='infer()', globals={'infer': infer}, num_threads=2).blocked_autorange(min_run_time=0.2)
print(measurement)


## Step 4 — Try mixed precision when CUDA is available

Autocast chooses lower precision for suitable operations; GradScaler reduces float16 gradient underflow. Unscale before gradient clipping. This optional branch demonstrates one real GPU training step; without CUDA it reports a skip and the CPU lessons remain complete.


In [ ]:
import os
RUN_CUDA = os.environ.get('PYTORCH_COURSE_CUDA', '0') == '1'  # Set True to try your CUDA GPU.
if RUN_CUDA and torch.cuda.is_available():
    gpu_model = copy.deepcopy(full).to('cuda').train()
    gpu_x, gpu_y = X.to('cuda'), y.to('cuda')
    gpu_optimizer = torch.optim.Adam(gpu_model.parameters(), lr=0.001)
    scaler = torch.amp.GradScaler('cuda')
    gpu_optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        gpu_loss = nn.functional.mse_loss(gpu_model(gpu_x), gpu_y)
    scaler.scale(gpu_loss).backward()
    scaler.unscale_(gpu_optimizer)
    nn.utils.clip_grad_norm_(gpu_model.parameters(), 1.)
    scaler.step(gpu_optimizer); scaler.update()
    torch.cuda.synchronize()
    assert torch.isfinite(gpu_loss)
    print('CUDA AMP step completed. Loss:', gpu_loss.item())
else:
    print('CUDA AMP skipped. Set RUN_CUDA=True if a compatible CUDA GPU is available.')


## Step 5 — Separate graph capture from optimized compilation

The eager backend demonstrates `torch.compile` graph capture without requiring a native compiler. It is not a speed optimization. The optional inductor backend generates optimized code and may need a platform-specific toolchain. Compile time can dominate tiny workloads; compare steady-state performance only after compilation and warmup.


In [ ]:
compiled_demo = torch.compile(full, backend='eager', fullgraph=True)
with torch.inference_mode():
    assert torch.allclose(compiled_demo(X), full(X), atol=1e-6)
print('Graph capture correctness check passed (eager backend).')
RUN_INDUCTOR = False  # Optional: requires a supported compiler/toolchain for your platform.
if RUN_INDUCTOR:
    optimized = torch.compile(full, backend='inductor')
    with torch.inference_mode():
        optimized(X)  # Compilation / warmup, excluded from timing.
        assert torch.allclose(optimized(X), full(X), atol=1e-5)
    @torch.inference_mode()
    def infer_optimized():
        return optimized(X)
    print(Timer(stmt='infer_optimized()', globals={'infer_optimized': infer_optimized},
                num_threads=2).blocked_autorange(min_run_time=0.2))
else:
    print('Optimized inductor compilation skipped; enable after installing its platform prerequisites.')


## Your turn — practice before opening the solutions

1. Repeat the accumulation comparison with microbatch size 5 and Adam.
2. Print the batch size that resulted from each split to confirm the final one is smaller.
3. Compute the maximum absolute prediction difference between the graph-captured and eager models.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
a, b = copy.deepcopy(full), copy.deepcopy(full)
oa, ob = torch.optim.Adam(a.parameters(), lr=0.001), torch.optim.Adam(b.parameters(), lr=0.001)
oa.zero_grad(set_to_none=True)
nn.functional.mse_loss(a(X), y).backward(); oa.step()
ob.zero_grad(set_to_none=True)
for xb, yb in zip(X.split(5), y.split(5)):
    (nn.functional.mse_loss(b(xb), yb) * len(xb) / len(X)).backward()
ob.step()
assert all(torch.allclose(pa, pb, atol=1e-6) for pa, pb in zip(a.parameters(), b.parameters()))
print('Microbatch sizes:', [len(xb) for xb in X.split(5)])
with torch.inference_mode():
    print('Maximum graph-capture difference:', (compiled_demo(X) - full(X)).abs().max().item())


## Before moving on

Measure throughput, peak memory, and accuracy together. Faster arithmetic is not useful if data loading dominates or numerical changes hurt the task. Use larger representative workloads before drawing conclusions about GPU or compilation speedups.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [AMP documentation](https://docs.pytorch.org/docs/stable/amp.html)
- [torch.compile tutorial](https://docs.pytorch.org/tutorials/intermediate/torch_compile_tutorial.html)
- [Profiler recipe](https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html)


[Course guide](../README.md) · [Previous: 16](16_custom_autograd_and_debugging.ipynb) · [Next: 18](18_distributed_training.ipynb)
